# Experiment 5 — Human Evaluation of Explanation Faithfulness and Readability

## 1. Research questions

This condition-blinded, single-reviewer evaluation compares baseline explanations (A), original SHAP-grounded drafts (B), and final verified deliveries (C). The same reviewer evaluates every record. The study uses saved outputs from Experiments 1–3 and compares human judgments with the frozen automated checks. Experiment 4 motivates verifier comparisons and contributes no primary rating records.

- **H5a.** B will have higher human-assessed faithfulness than A. The human rubric independently judges C against the saved reference.
- **H5b.** C may have lower readability than B when deterministic fallback introduces repetitive phrasing.
- **H5c.** Faithfulness and readability may improve or decline separately.
- **H5d.** In the optional synthetic study, explanations may remain clinically plausible while disagreeing with a counterintuitive model reference.

## 2. Frozen experimental sources

The primary study uses the saved Experiment 2 checkpoint and the pinned Experiment 1 model bundle containing probabilities and signed SHAP contributions. The optional secondary study uses the saved Experiment 3 checkpoint and its pinned scenario manifest. C may deliver the original B draft, an accepted Qwen3 revision or a deterministic fallback. The final delivered C text is rated without its delivery metadata. Both checkpoints are opened read-only.

## 3. Sampling and pairing

Select eight complete A/B/C admission triplets per frozen risk stratum, for 24 admissions and 72 unique condition records. The fixed-seed sample uses all eligible cases as its frame and logs each unusable source output before selection. Optional synthetic sampling selects up to four profiles per family and keeps both model variants together. Tracker counts below distinguish the sampling frame, selected records and completed ratings.

## 4. Blinding and reviewer protocol

Use the separate local review command in the [reviewer guide](../experiment5/README.md). The terminal presents randomized blind IDs and explanation prose. Record clarity and readability before opening the reference. Evaluate the saved probability and signed SHAP contributions for faithfulness and completeness, then record uncertainty, unsupported claims and brief evidence notes. Condition labels, verifier judgments, and delivery source stay in the private investigator crosswalk.

## 5. Rubric

Use the [frozen rubric](../experiment5/rubric.md), [protocol](../experiment5/protocol.md) and [fictional calibration set](../experiment5/calibration.md). Faithfulness has four categories, including indeterminate and invalid. Completeness and uncertainty have their own categories. Clarity and readability are separate 1–5 scales.

## 6. Dataset preparation

Setup validates the pinned sources and selects records. Set `WRITE_WORKSHEETS` explicitly to create owner-only review files outside Git. `review` is read-only, `analyze` computes results from locked primary annotations, and `fictional` uses scripted software fixtures. Keep the worksheet, references and crosswalk out of notebook output.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

for candidate in (Path.cwd(), Path.cwd() / 'src', Path.cwd() / 'HRP' / 'src'):
    if (candidate / 'human_explanation_evaluation.py').is_file():
        sys.path.insert(0, str(candidate))
        break

from human_explanation_evaluation import notebook_run
from human_evaluation_figures import build_figures

MODE = 'setup'
PRIVATE_DIRECTORY = None  # The default is documented in the reviewer guide.
WRITE_WORKSHEETS = False
RUN_SYNTHETIC = False
SYNTHETIC_DIRECTORY = None

try:
    run = notebook_run(MODE, directory=PRIVATE_DIRECTORY, write_worksheets=WRITE_WORKSHEETS)
except FileNotFoundError:
    run = {'status': 'Awaiting human review', 'detail': 'The frozen input bundle or private study files are unavailable.'}
result = run.get('result')
print(run['status'])
if 'detail' in run:
    print(run['detail'])
if 'accounting' in run:
    display(pd.DataFrame.from_dict(run['accounting'], orient='index'))
    display({key: run[key] for key in ('source_sha256', 'selected_by_condition', 'excluded_source_outputs')})

## 7. Annotation completion

The primary pass has 72 expected ratings from one reviewer. Each completed row is validated before storage. Locked annotations and source provenance are verified before the analysis links human judgments to the conditions. Missing repeat rows appear in the status and results.

In [ ]:
if 'primary_target' in run:
    display(pd.DataFrame({
        'Target': [run['primary_target'], run['repeat_target']],
        'Selected': [run['primary_selected'], run['repeat_selected']],
        'Completed': [run['primary_completed'], run['repeat_completed']],
        'Pending in packet': [run['primary_pending'], run['repeat_pending']],
    }, index=['Primary', 'Repeat']))
    print(f"Planned reviewers: {run['reviewers']}. Primary ratings locked: {run['first_pass_locked']}.")

## 8. Intra-reviewer repeat consistency

A separate pass presents four records per condition with new blind IDs and hides initial ratings. Prefer a gap of at least three days. Report exact faithfulness and completeness agreement, absolute clarity and readability differences, percentages above one point, and quadratic-weighted Cohen’s kappa for sufficiently varied ordinal ratings. Repeat judgments enter only this analysis. The repeat measure concerns rating stability within this reviewer.

In [ ]:
if result and 'repeat_consistency' in result:
    display(result['repeat_consistency'])

## 9. Mathematical faithfulness and completeness

Report all categorical counts, determinate and selected denominators, and matched transitions for B minus A, C minus B and C minus A. Keep indeterminate and invalid ratings visible.

## 10. Clarity and readability

Report each scale’s ordinal distribution, descriptive mean, median and standard deviation alongside paired changes.

## 11. Faithfulness and readability

Compare readability and clarity among faithful and unfaithful judgments within each condition. Interpret associations descriptively.

In [ ]:
if result:
    display(pd.DataFrame.from_dict(result['by_condition'], orient='index'))
    display(result['paired'])
    display(result['ratings_by_faithfulness'])

## 12. Condition C delivery source

Compare original grounded draft acceptance, accepted revision, and deterministic fallback descriptively. The verifier selects the delivery groups and may encounter different case difficulty in each.

## 13. Human judgments and the frozen verifier

Cross-tabulate human faithfulness against structured attribution correctness, narrative flags and final C release status. Preserve human indeterminate judgments and the verifier's ambiguous findings.

## 14. Risk-stratified results

Report human ratings separately for the low-, medium- and high-predicted-risk groups.

In [ ]:
if result:
    display(result['condition_c_by_final_source'])
    display(result['human_vs_frozen_verifier'])
    if 'by_risk_group' in result:
        display(result['by_risk_group'])
    for figure in build_figures(result).values():
        display(figure)
        plt.close(figure)

## 15. Optional synthetic study

Compare human ratings of intuitive and counterintuitive variants within profile. Retain both variants, all three conditions and each variant’s own reference. The synthetic analysis has its own annotation counts and aggregates.

In [ ]:
if RUN_SYNTHETIC or MODE == 'fictional':
    if MODE == 'fictional':
        secondary = {'status': run['status'], 'result': run['synthetic_result']}
    else:
        try:
            secondary = notebook_run(MODE, directory=SYNTHETIC_DIRECTORY, study='synthetic',
                                     write_worksheets=WRITE_WORKSHEETS)
        except FileNotFoundError:
            secondary = {'status': 'Awaiting synthetic source files'}
    print(secondary['status'])
    if 'accounting' in secondary:
        display(pd.DataFrame.from_dict(secondary['accounting'], orient='index'))
        print(f"Synthetic ratings: {secondary['primary_completed']} / {secondary['primary_target']}")
    if 'result' in secondary:
        display(secondary['result']['synthetic_by_family_and_variant'])
        display(secondary['result']['variant_pairs'])
else:
    print('Optional synthetic review is disabled.')

## 16. Illustrative examples

Export selects observed examples when they meet prespecified criteria. The private ID index links to the checked worksheets, references and notes. Resolve those links only in the private study folder. Public illustration should use reviewed fictional examples after the optional synthetic study is complete.

## 17. Statistical uncertainty

Use 10,000 fixed-seed bootstrap replicates with whole admissions or fictional profiles as the unit. Paired comparisons preserve each case’s condition ratings. Ordinal changes are descriptive, and small denominators may limit the stability of intervals. Reports show eligible denominators alongside selected counts. Use the [reviewer guide](../experiment5/README.md) for versioned aggregate and figure export.

## 18. Discussion and limitations

The single reviewer has prior familiarity with the project and the explanation framework. No inter-reviewer reliability estimate is available. Expectations and recognizable deterministic fallback wording may influence judgments despite hidden labels. Repeat consistency measures within-reviewer stability. Correctness, external validity, and clinical usability require separate evidence. The exploratory ratings represent one person’s judgments about saved model explanations.

## 19. Research synthesis handoff

Once review is complete, retain matched denominators, the direction and uncertainty of contrasts, C source counts, human–verifier disagreements and repeat consistency. Report optional synthetic results separately. Leave hypotheses open while genuine review is pending.